In [0]:
tabelas = [
    "workspace.default.bronze_anamnese_raw",
    "workspace.default.dim_atleta",
    "workspace.default.dim_condicao_clinica",
    "workspace.default.dim_desempenho_corrida",
    "workspace.default.dim_atleta_trainingpeaks",
    "workspace.default.fato_sessao_treino",
    "workspace.default.gold_prescricao_diaria"
]

for tabela in tabelas:
    try:
        df = spark.table(tabela)
        print(f"OK | {tabela} | registros: {df.count()}")
        print("Colunas:", df.columns)
    except Exception as erro:
        print(f"NÃO ENCONTRADA ou com erro | {tabela}")
        print(str(erro).split("\n")[0])

OK | workspace.default.bronze_anamnese_raw | registros: 4
Colunas: ['carimbo_de_data_hora', 'nome_completo', 'e_mail_de_contato', 'whatsapp_com_ddd', 'data_de_nascimento', 'sexo_biologico', 'termo_de_consentimento_e_uso_de_dados_lgpd', 'altura_cm', 'peso_corporal_atual_em_kg', 'qual_o_seu_objetivo_principal_no_momento', 'tem_alguma_prova_alvo_ja_agendada_se_sim_qual_e_qu', 'quantos_dias_por_semana_voce_costuma_treinar_e_qua', 'em_qual_horario_voce_treina_habitualmente_e_em_qua', 'utiliza_o_trainingpeaks_se_sim_insira_seu_link_usu', 'voce_possui_ou_ja_teve_alguma_das_seguintes_condic', 'se_marcou_alguma_condicao_acima_ou_outra_condicao_', 'utiliza_medicamentos_de_uso_continuo', 'liste_os_medicamentos_de_uso_continuo_informando_n', 'utiliza_medicamentos_ou_produtos_de_forma_ocasiona', 'possui_alergias_ou_intolerancias', 'existe_historico_familiar_de_doenca_cardiovascular', 'durante_ou_apos_o_exercicio_ja_apresentou_algum_de', 'se_apresentou_algum_sintoma_descreva_quando_ocorre', 'quais_s

In [0]:
from pyspark.sql import functions as F

df_bronze = spark.table("workspace.default.bronze_anamnese_raw")

# Detecta automaticamente a coluna de e-mail de contato (evita erro de nome)
candidatas = [c for c in df_bronze.columns if "mail" in c and "contato" in c]
coluna_email = candidatas[0]
print("Coluna de e-mail identificada:", coluna_email)

# ---------- dim_atleta ----------
df_atleta = (
    df_bronze
    .withColumn("id_atleta", F.sha2(F.lower(F.trim(F.col(coluna_email))), 256))
    .select(
        F.col("id_atleta"),
        F.col("data_de_nascimento").alias("data_nascimento"),
        F.col("sexo_biologico").alias("sexo"),
        F.col("altura_cm").cast("double").alias("altura_cm"),
        F.col("peso_corporal_atual_em_kg").cast("double").alias("peso_kg"),
        F.col("qual_o_seu_objetivo_principal_no_momento").alias("objetivo_principal"),
        F.col("tem_alguma_prova_alvo_ja_agendada_se_sim_qual_e_qu").alias("prova_alvo"),
        F.col("quantos_dias_por_semana_voce_costuma_treinar_e_qua").alias("frequencia_treino_semanal"),
        F.col("em_qual_horario_voce_treina_habitualmente_e_em_qua").alias("horario_treino"),
        F.col("utiliza_o_trainingpeaks_se_sim_insira_seu_link_usu").alias("trainingpeaks_link"),
        F.col("data_ingestao")
    )
    .withColumn(
        "idade_anos",
        F.floor(F.months_between(F.current_date(), F.to_date(F.col("data_nascimento"), "dd/MM/yyyy")) / 12)
    )
    .dropDuplicates(["id_atleta"])
)
df_atleta.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.default.dim_atleta")
print("dim_atleta atualizada. Registros:", df_atleta.count())

# ---------- dim_condicao_clinica ----------
df_condicao_clinica = (
    df_bronze
    .withColumn("id_atleta", F.sha2(F.lower(F.trim(F.col(coluna_email))), 256))
    .select(
        F.col("id_atleta"),
        F.col("voce_possui_ou_ja_teve_alguma_das_seguintes_condic").alias("condicoes_clinicas"),
        F.col("se_marcou_alguma_condicao_acima_ou_outra_condicao_").alias("condicoes_clinicas_detalhe"),
        F.col("utiliza_medicamentos_de_uso_continuo").alias("usa_medicamento_continuo"),
        F.col("liste_os_medicamentos_de_uso_continuo_informando_n").alias("medicamentos_continuos_detalhe"),
        F.col("utiliza_medicamentos_ou_produtos_de_forma_ocasiona").alias("medicamentos_ocasionais_detalhe"),
        F.col("possui_alergias_ou_intolerancias").alias("alergias_intolerancias"),
        F.col("existe_historico_familiar_de_doenca_cardiovascular").alias("historico_familiar_cardiovascular"),
        F.col("durante_ou_apos_o_exercicio_ja_apresentou_algum_de").alias("sintomas_durante_exercicio"),
        F.col("se_apresentou_algum_sintoma_descreva_quando_ocorre").alias("sintomas_detalhe"),
        F.col("voce_possui_teste_de_ergoespirometria_ou_teste_car").alias("possui_teste_ergoespirometria"),
        F.col("se_sim_envie_o_pdf_do_exame_35").alias("link_pdf_ergoespirometria"),
        F.col("voce_possui_exames_laboratoriais_recentes").alias("possui_exames_laboratoriais"),
        F.col("se_sim_envie_o_pdf_do_exame_37").alias("link_pdf_exames_laboratoriais"),
        F.col("voce_autoriza_o_alinhamento_das_suas_orientacoes_d").alias("autoriza_compartilhamento_orientacoes"),
        F.col("data_ingestao")
    )
    .dropDuplicates(["id_atleta"])
)
df_condicao_clinica.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.default.dim_condicao_clinica")
print("dim_condicao_clinica atualizada. Registros:", df_condicao_clinica.count())

# ---------- dim_desempenho_corrida ----------
coluna_tempo_5k = [c for c in df_bronze.columns if "5_km" in c or "5k" in c.lower()][0]
coluna_tempo_10k = [c for c in df_bronze.columns if "10_km" in c or "10k" in c.lower()][0]
coluna_origem_5k = "origem_do_tempo_informado_nos_5_km"
coluna_origem_10k = "origem_do_tempo_informado_nos_10_km"

df_corrida = df_bronze.withColumn("id_atleta", F.sha2(F.lower(F.trim(F.col(coluna_email))), 256))

def converter_tempo_para_segundos(nome_coluna):
    valor = F.trim(F.col(nome_coluna))
    partes = F.split(valor, ":")
    return (
        F.when(F.col(nome_coluna).isNull() | (valor == ""), F.lit(None).cast("double"))
        .when(F.size(partes) == 2, partes.getItem(0).cast("double") * 60 + partes.getItem(1).cast("double"))
        .when(F.size(partes) == 3, partes.getItem(0).cast("double") * 3600 + partes.getItem(1).cast("double") * 60 + partes.getItem(2).cast("double"))
        .otherwise(F.lit(None).cast("double"))
    )

df_desempenho_corrida = (
    df_corrida
    .select(
        "id_atleta",
        F.col(coluna_tempo_5k).alias("tempo_5k_bruto"),
        F.col(coluna_origem_5k).alias("origem_tempo_5k"),
        F.col(coluna_tempo_10k).alias("tempo_10k_bruto"),
        F.col(coluna_origem_10k).alias("origem_tempo_10k")
    )
    .withColumn("tempo_5k_segundos", converter_tempo_para_segundos("tempo_5k_bruto"))
    .withColumn("tempo_10k_segundos", converter_tempo_para_segundos("tempo_10k_bruto"))
    .withColumn("tempo_5k_minutos", F.round(F.col("tempo_5k_segundos") / 60, 2))
    .withColumn("tempo_10k_minutos", F.round(F.col("tempo_10k_segundos") / 60, 2))
    .withColumn(
        "metodo_avaliacao",
        F.when(F.col("tempo_5k_segundos").isNotNull() | F.col("tempo_10k_segundos").isNotNull(), F.lit("TESTE_DE_CAMPO"))
        .otherwise(F.lit("NAO_INFORMADO"))
    )
    .dropDuplicates(["id_atleta"])
)
df_desempenho_corrida.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.default.dim_desempenho_corrida")
print("dim_desempenho_corrida atualizada. Registros:", df_desempenho_corrida.count())

Coluna de e-mail identificada: e_mail_de_contato
dim_atleta atualizada. Registros: 2
dim_condicao_clinica atualizada. Registros: 2
dim_desempenho_corrida atualizada. Registros: 2


In [0]:
df_bronze = spark.table("workspace.default.bronze_anamnese_raw")

df_bronze.select("nome_completo", "e_mail_de_contato").show(truncate=False)

+-------------------------------+---------------------------+
|nome_completo                  |e_mail_de_contato          |
+-------------------------------+---------------------------+
|Marcelo Albuquerque de Oliveira|farma.albuquerque@gmail.com|
|Marcelo Albuquerque de Oliveira|farma.albuquerque@gmail.com|
|Douglas Bordalo da Silva       |NULL                       |
|Rafael Souto de Souza          |NULL                       |
+-------------------------------+---------------------------+

